# 45 · 前缀建议、相关查询与会话推荐

单元：S16。先修：38、39、43。

来源：[王树森固定目录](https://github.com/wangshusen/SearchEngine/tree/7339c6b126e314abdb74a89adb033d39d8e3d37b)。这是 Lab 自编实验；王树森后四模块只有公开目录，不称原作者完整讲义。

证据边界：完全构造查询日志；无真实用户或隐私数据。

学习方式：先预测每个对照，再只改一个变量，运行后解释差异。

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
from reclab.learning.search_curriculum import *

## 三个任务
前缀补全必须满足前缀；相关query服务当前需求；下一query预测依赖会话。先定义输入输出，再用频次和转移生成候选，不能以点击热门词代表相关性。

In [2]:
print('CONSTRUCTED sessions',SESSIONS)
print('prefix apple',suggestions('apple'))
print('related apple repair',related('apple repair'))
assert all(q.startswith('apple') for q in suggestions('apple'))

CONSTRUCTED sessions [('apple repair', 'phone battery repair'), ('apple repair', 'apple official repair'), ('apple fruit', 'apple pie'), ('apple repair', 'phone battery repair'), ('phone battery repair', 'battery price'), ('apple fruit', 'apple pie')]
prefix apple ['apple repair', 'apple fruit', 'apple pie', 'apple official repair']
related apple repair ['phone battery repair', 'apple official repair']


## 训练和保留会话
按会话边界拆分；重复模板会泄漏，此例故意展示模板重合，不报告泛化成绩。固定前四条fit，后两条机制评估，列出重合而非掩盖。

In [3]:
fit=SESSIONS[:4]; hold=SESSIONS[4:]
print('template overlap',set(fit)&set(hold))
rows=[]
for a,b in hold:
    pred=related(a,fit); rank=pred.index(b)+1 if b in pred else None
    rows.append({'query':a,'next':b,'pred':pred,'RR':1/rank if rank else 0})
display(pd.DataFrame(rows))

template overlap {('apple fruit', 'apple pie')}


,query,next,pred,RR
0,phone battery repair,battery price,[],0.0
1,apple fruit,apple pie,[apple pie],1.0


## 前缀排序与相关性约束
候选先按前缀召回再频次排序。前缀无结果显式返回空；不是偷偷改前缀。只改变prefix。

In [4]:
for prefix in ['apple','apple r','unknown']:
    pred=suggestions(prefix,fit); print(prefix,pred)
    assert all(q.startswith(prefix) for q in pred)
assert suggestions('unknown',fit)==[]
# 人工前缀判断：相关集合只对当前补全任务有效。
gold={'apple repair','apple official repair'}
ranked=suggestions('apple',fit)
for k in [1,3]:
    head=ranked[:k]
    print('prefix k',k,'precision',len(set(head)&gold)/max(len(head),1),'recall',len(set(head)&gold)/len(gold))


apple ['apple repair', 'apple fruit', 'apple official repair', 'apple pie']
apple r ['apple repair']
unknown []
prefix k 1 precision 1.0 recall 0.5
prefix k 3 precision 0.6666666666666666 recall 1.0


## 漂移与回退
会话需求可能从维修转价格。只返回同锚词是保守机制，会漏掉合理转移；展示coverage损失。实际需要人工相关性、覆盖、时效及线上拒用率评估。

In [5]:
for q in ['apple repair','phone battery repair','unknown']:
    raw=related(q); safe=[x for x in raw if set(terms(q))&set(terms(x))]
    print(q,'raw',raw,'guarded',safe,'fallback prefix',suggestions(q) if not safe else [])

apple repair raw ['phone battery repair', 'apple official repair'] guarded ['phone battery repair', 'apple official repair'] fallback prefix []
phone battery repair raw ['battery price'] guarded ['battery price'] fallback prefix []
unknown raw [] guarded [] fallback prefix []


## 本次保存结果的解读

保留会话中一个转移无候选（RR=0），另一个命中（RR=1）却与fit模板重复，故不能称可靠泛化。人工前缀判断在k=1时precision=1、recall=0.5，k=3时precision约0.667、recall=1，体现覆盖与精度取舍。unknown显式返回空结果。

改参数后应以新输出重新判断，不把这段固定解读当作新的测量。

## 复盘与练习

写下输入、输出、保持不变的量与本章不能支持的结论。选一个上面的单变量参数先预测再重跑；保留失败例。回到搜索学习路径继续下一语义单元。